# Batch image censoring with the Puryfi model (PyTorch)

This notebook rebuilds the batch image censoring pipeline using **PyTorch** only. It loads the TensorFlow.js graph model that ships with the Firefox extension, converts the weights into a PyTorch friendly format, and then applies the detector to local images so they can be censored in bulk.


## 1. Install the required Python packages

Run the following cell if your environment does not already provide PyTorch, Pillow, NumPy, Matplotlib, and tqdm.


In [ ]:
# Uncomment the line below if you need to install the runtime dependencies.
# %pip install torch torchvision pillow numpy matplotlib tqdm


## 2. Import standard-library helpers


In [ ]:
from __future__ import annotations

import json
import math
import re
import struct
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple


## 3. Import third-party libraries


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFilter
from tqdm.auto import tqdm


## 4. Define extension and model paths


In [ ]:
BASE_DIR = Path.cwd()
EXTENSION_DIR = BASE_DIR / "releases" / "puryfi-0.8.0.8-anfx"
MODEL_DIR = EXTENSION_DIR / "web_model"
MODEL_JSON = MODEL_DIR / "model.json"
print("Extension directory:", EXTENSION_DIR)
print("Model directory:", MODEL_DIR)


## 5. Record the detection label ordering

The Firefox extension defines 26 label keys, but the shipped model only predicts the first 20 indices (0-19). We parse the bundle once so the detector can translate class indices back to human readable labels.


In [ ]:
KLASS_SOURCE = EXTENSION_DIR / "core" / "puryfi-core-context.bundle.js"
pattern = re.compile(r"([A-Z0-9_]+): \{[^}]*?key: \"([A-Z0-9]+)\"[^}]*?index: (\d+)")
klass_entries = []
for match in pattern.finditer(KLASS_SOURCE.read_text()):
    display_name, key, index = match.groups()
    klass_entries.append((int(index), key))
klass_entries.sort()
ALL_LABEL_KEYS = [key for index, key in klass_entries]
MODEL_LABEL_KEYS = ALL_LABEL_KEYS[:20]
print("Detected label keys (first 20 used by the detector):")
for idx, key in enumerate(MODEL_LABEL_KEYS):
    print(f"{idx:>2}: {key}")


## 6. Inspect the TensorFlow.js manifest


In [ ]:
with MODEL_JSON.open() as f:
    tfjs_model = json.load(f)
manifest = tfjs_model["weightsManifest"][0]
print("Binary shards:", manifest["paths"])
print("Total tensors described:", len(manifest["weights"]))
print("Input placeholder nodes:")
for node in tfjs_model["modelTopology"]["node"]:
    if node["op"] == "Placeholder":
        print("  ", node["name"], node["attr"]["shape"])


## 7. Load TensorFlow.js graph weights

TensorFlow.js stores all trainable tensors inside a series of binary shards. The helper below reconstructs each tensor into a NumPy array so we can reuse the data from PyTorch.


In [ ]:
def load_tfjs_graph_weights(model_dir: Path) -> Dict[str, np.ndarray]:
    model_json = json.loads((model_dir / "model.json").read_text())
    manifest = model_json["weightsManifest"][0]
    shards = [(model_dir / path).read_bytes() for path in manifest["paths"]]
    buffer = b"".join(shards)
    offset = 0
    weights: Dict[str, np.ndarray] = {}
    for entry in manifest["weights"]:
        name = entry["name"]
        shape = [int(dim) for dim in entry["shape"]]
        dtype = entry["dtype"]
        count = int(np.prod(shape, dtype=np.int64)) if shape else 1
        if dtype == "float32":
            array = np.frombuffer(buffer, dtype=np.float32, count=count, offset=offset)
            itemsize = 4
        elif dtype == "int32":
            array = np.frombuffer(buffer, dtype=np.int32, count=count, offset=offset)
            itemsize = 4
        else:
            raise ValueError(f"Unsupported dtype: {dtype}")
        weights[name] = array.reshape(shape)
        offset += count * itemsize
    if offset != len(buffer):
        raise RuntimeError("Weights manifest parsing mismatch")
    return weights

TF_WEIGHTS = load_tfjs_graph_weights(MODEL_DIR)
print(f"Loaded {len(TF_WEIGHTS)} tensors from the TensorFlow.js manifest")
example_name = next(iter(TF_WEIGHTS))
print("Example tensor:", example_name, TF_WEIGHTS[example_name].shape)


## 8. Define PyTorch building blocks

The detector follows the YOLOv5 architecture. We recreate the lightweight Focus, Conv, C3, Bottleneck, and SPPF blocks that the original model used.


In [ ]:
def autopad(kernel_size: int, stride: int = 1) -> int:
    return (kernel_size - 1) // 2 if stride == 1 else 0


class ConvBNAct(nn.Module):
    def __init__(self, c1: int, c2: int, k: int = 1, s: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, kernel_size=k, stride=s, padding=autopad(k, s), bias=True)
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.conv(x))


class Focus(nn.Module):
    def __init__(self, c1: int, c2: int, k: int = 3):
        super().__init__()
        self.conv = ConvBNAct(c1 * 4, c2, k, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        patches = torch.cat((x[..., ::2, ::2], x[..., 1::2, ::2], x[..., ::2, 1::2], x[..., 1::2, 1::2]), dim=1)
        return self.conv(patches)


class Bottleneck(nn.Module):
    def __init__(self, c1: int, c2: int, shortcut: bool = True):
        super().__init__()
        self.cv1 = ConvBNAct(c1, c2, 1, 1)
        self.cv2 = ConvBNAct(c2, c2, 3, 1)
        self.shortcut = shortcut and c1 == c2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.cv2(self.cv1(x))
        return x + y if self.shortcut else y


class C3(nn.Module):
    def __init__(self, c1: int, c2: int, n: int = 1, shortcut: bool = True):
        super().__init__()
        hidden = c2 // 2
        self.cv1 = ConvBNAct(c1, hidden, 1, 1)
        self.cv2 = ConvBNAct(c1, hidden, 1, 1)
        self.cv3 = ConvBNAct(hidden * 2, c2, 1, 1)
        self.m = nn.ModuleList([Bottleneck(hidden, hidden, shortcut) for _ in range(n)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y1 = self.cv1(x)
        for bottleneck in self.m:
            y1 = bottleneck(y1)
        y2 = self.cv2(x)
        return self.cv3(torch.cat((y1, y2), dim=1))


class SPPF(nn.Module):
    def __init__(self, c1: int, c2: int, k: int = 5):
        super().__init__()
        hidden = c1 // 2
        self.cv1 = ConvBNAct(c1, hidden, 1, 1)
        self.cv2 = ConvBNAct(hidden * 4, c2, 1, 1)
        self.k = k

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.cv1(x)
        y1 = F.max_pool2d(x, kernel_size=self.k, stride=1, padding=self.k // 2)
        y2 = F.max_pool2d(y1, kernel_size=self.k, stride=1, padding=self.k // 2)
        y3 = F.max_pool2d(y2, kernel_size=self.k, stride=1, padding=self.k // 2)
        return self.cv2(torch.cat((x, y1, y2, y3), dim=1))


## 9. Build the detection head and model wrapper

We implement a Detect module that mirrors YOLOv5's decode logic (including anchor grids and strides) and a convenience wrapper that wires all blocks together.


In [ ]:
class Detect(nn.Module):
    def __init__(self, num_classes: int, anchors: torch.Tensor, strides: torch.Tensor):
        super().__init__()
        self.nc = num_classes
        self.no = num_classes + 5
        self.nl = anchors.shape[0]
        self.na = anchors.shape[1]
        self.register_buffer("anchors", anchors.clone())
        self.register_buffer("anchor_grid", anchors.clone().view(self.nl, 1, self.na, 1, 1, 2))
        self.register_buffer("stride", strides.clone())
        self.m = nn.ModuleList([nn.Conv2d(ch, self.na * self.no, 1) for ch in (128, 256, 512)])
        self._grid_cache: Dict[Tuple[int, int, int], torch.Tensor] = {}

    def _make_grid(self, layer: int, nx: int, ny: int, device: torch.device) -> torch.Tensor:
        key = (layer, ny, nx)
        if key not in self._grid_cache:
            yv, xv = torch.meshgrid(torch.arange(ny, device=device), torch.arange(nx, device=device), indexing="ij")
            self._grid_cache[key] = torch.stack((xv, yv), dim=2).view(1, 1, ny, nx, 2).float()
        return self._grid_cache[key]

    def forward(self, features: Sequence[torch.Tensor]) -> torch.Tensor:
        outputs = []
        for layer, (conv, x) in enumerate(zip(self.m, features)):
            pred = conv(x)
            bs, _, ny, nx = pred.shape
            pred = pred.view(bs, self.na, self.no, ny, nx).permute(0, 1, 3, 4, 2).contiguous()
            pred = pred.sigmoid()
            grid = self._make_grid(layer, nx, ny, pred.device)
            xy = (pred[..., 0:2] * 2.0 - 0.5 + grid) * self.stride[layer]
            wh = (pred[..., 2:4] * 2.0) ** 2 * self.anchor_grid[layer]
            obj = pred[..., 4:5]
            cls = pred[..., 5:]
            decoded = torch.cat((xy, wh, obj, cls), dim=-1)
            outputs.append(decoded.view(bs, -1, self.no))
        return torch.cat(outputs, dim=1)


class PuryfiYoloV5(nn.Module):
    def __init__(self, num_classes: int = 20):
        super().__init__()
        self.focus = Focus(3, 32)
        self.conv1 = ConvBNAct(32, 64, 3, 2)
        self.c3_0 = C3(64, 64, n=1, shortcut=True)
        self.conv2 = ConvBNAct(64, 128, 3, 2)
        self.c3_1 = C3(128, 128, n=3, shortcut=True)
        self.conv3 = ConvBNAct(128, 256, 3, 2)
        self.c3_2 = C3(256, 256, n=3, shortcut=True)
        self.conv4 = ConvBNAct(256, 512, 3, 2)
        self.sppf = SPPF(512, 512, 5)
        self.c3_3 = C3(512, 512, n=1, shortcut=False)
        self.conv5 = ConvBNAct(512, 256, 1, 1)
        self.c3_4 = C3(512, 256, n=1, shortcut=False)
        self.conv6 = ConvBNAct(256, 128, 1, 1)
        self.c3_5 = C3(256, 128, n=1, shortcut=False)
        self.conv7 = ConvBNAct(128, 128, 3, 2)
        self.c3_6 = C3(256, 256, n=1, shortcut=False)
        self.conv8 = ConvBNAct(256, 256, 3, 2)
        self.c3_7 = C3(512, 512, n=1, shortcut=False)
        anchors = torch.zeros((3, 3, 2), dtype=torch.float32)
        strides = torch.zeros(3, dtype=torch.float32)
        self.detect = Detect(num_classes, anchors, strides)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x0 = self.focus(x)
        x1 = self.c3_0(self.conv1(x0))
        x2 = self.c3_1(self.conv2(x1))
        x3 = self.c3_2(self.conv3(x2))
        x4 = self.c3_3(self.sppf(self.conv4(x3)))

        p5_reduced = self.conv5(x4)
        up1 = F.interpolate(p5_reduced, scale_factor=2, mode="nearest")
        y1 = self.c3_4(torch.cat((up1, x3), dim=1))

        p4_reduced = self.conv6(y1)
        up2 = F.interpolate(p4_reduced, scale_factor=2, mode="nearest")
        y2 = self.c3_5(torch.cat((up2, x2), dim=1))

        p3_down = self.conv7(y2)
        y3 = self.c3_6(torch.cat((p3_down, p4_reduced), dim=1))

        p4_down = self.conv8(y3)
        y4 = self.c3_7(torch.cat((p4_down, p5_reduced), dim=1))

        return self.detect([y2, y3, y4])


## 10. Map TensorFlow.js weights into the PyTorch model

The following helpers translate TensorFlow.js tensor names into the PyTorch module hierarchy. Convolution kernels are converted from `[H, W, C_in, C_out]` (TensorFlow) to `[C_out, C_in, H, W]` (PyTorch).


In [ ]:
def assign_conv(module: ConvBNAct, prefix: str, source: Dict[str, np.ndarray]) -> None:
    weight = source[f"{prefix}/Conv2D_weights"]
    bias = source[f"{prefix}/Conv2D_bn_offset"]
    module.conv.weight.data.copy_(torch.from_numpy(np.transpose(weight, (3, 2, 0, 1))))
    module.conv.bias.data.copy_(torch.from_numpy(bias.astype(np.float32)))


def assign_c3(block: C3, prefix: str, cv1: str, cv2: str, cv3: str, bottlenecks: List[Tuple[str, str, str]], source: Dict[str, np.ndarray]) -> None:
    assign_conv(block.cv1, f"{prefix}/{cv1}", source)
    assign_conv(block.cv2, f"{prefix}/{cv2}", source)
    for module, (sub_prefix, conv1, conv2) in zip(block.m, bottlenecks):
        assign_conv(module.cv1, f"{prefix}/{sub_prefix}/{conv1}", source)
        assign_conv(module.cv2, f"{prefix}/{sub_prefix}/{conv2}", source)
    assign_conv(block.cv3, f"{prefix}/{cv3}", source)


def assign_detect(conv: nn.Conv2d, prefix: str, source: Dict[str, np.ndarray]) -> None:
    weight = source[f"{prefix}/Conv2D/ReadVariableOp"]
    bias = source[f"{prefix}/BiasAdd/ReadVariableOp"]
    conv.weight.data.copy_(torch.from_numpy(np.transpose(weight, (3, 2, 0, 1))))
    conv.bias.data.copy_(torch.from_numpy(bias.astype(np.float32)))


def load_puryfi_weights(model: PuryfiYoloV5, source: Dict[str, np.ndarray]) -> None:
    assign_conv(model.focus.conv, "model/tf__focus/tf__conv/conv2d", source)
    assign_conv(model.conv1, "model/tf__conv_1/sequential/conv2d_1", source)
    assign_c3(model.c3_0, "model/tf__c3", "tf__conv_2/conv2d_2", "tf__conv_3/conv2d_3", "tf__conv_4/conv2d_4", [
        ("sequential_1/tf__bottleneck", "tf__conv_5/conv2d_5", "tf__conv_6/conv2d_6"),
    ], source)
    assign_conv(model.conv2, "model/tf__conv_7/sequential_2/conv2d_7", source)
    assign_c3(model.c3_1, "model/tf__c3_1", "tf__conv_8/conv2d_8", "tf__conv_9/conv2d_9", "tf__conv_10/conv2d_10", [
        ("sequential_3/tf__bottleneck_1", "tf__conv_11/conv2d_11", "tf__conv_12/conv2d_12"),
        ("sequential_3/tf__bottleneck_2", "tf__conv_13/conv2d_13", "tf__conv_14/conv2d_14"),
        ("sequential_3/tf__bottleneck_3", "tf__conv_15/conv2d_15", "tf__conv_16/conv2d_16"),
    ], source)
    assign_conv(model.conv3, "model/tf__conv_17/sequential_4/conv2d_17", source)
    assign_c3(model.c3_2, "model/tf__c3_2", "tf__conv_18/conv2d_18", "tf__conv_19/conv2d_19", "tf__conv_20/conv2d_20", [
        ("sequential_5/tf__bottleneck_4", "tf__conv_21/conv2d_21", "tf__conv_22/conv2d_22"),
        ("sequential_5/tf__bottleneck_5", "tf__conv_23/conv2d_23", "tf__conv_24/conv2d_24"),
        ("sequential_5/tf__bottleneck_6", "tf__conv_25/conv2d_25", "tf__conv_26/conv2d_26"),
    ], source)
    assign_conv(model.conv4, "model/tf__conv_27/sequential_6/conv2d_27", source)
    assign_conv(model.sppf.cv1, "model/tf_spp/tf__conv_28/conv2d_28", source)
    assign_conv(model.sppf.cv2, "model/tf_spp/tf__conv_29/conv2d_29", source)
    assign_c3(model.c3_3, "model/tf__c3_3", "tf__conv_30/conv2d_30", "tf__conv_31/conv2d_31", "tf__conv_32/conv2d_32", [
        ("sequential_7/tf__bottleneck_7", "tf__conv_33/conv2d_33", "tf__conv_34/conv2d_34"),
    ], source)
    assign_conv(model.conv5, "model/tf__conv_35/conv2d_35", source)
    assign_c3(model.c3_4, "model/tf__c3_4", "tf__conv_36/conv2d_36", "tf__conv_37/conv2d_37", "tf__conv_38/conv2d_38", [
        ("sequential_8/tf__bottleneck_8", "tf__conv_39/conv2d_39", "tf__conv_40/conv2d_40"),
    ], source)
    assign_conv(model.conv6, "model/tf__conv_41/conv2d_41", source)
    assign_c3(model.c3_5, "model/tf__c3_5", "tf__conv_42/conv2d_42", "tf__conv_43/conv2d_43", "tf__conv_44/conv2d_44", [
        ("sequential_9/tf__bottleneck_9", "tf__conv_45/conv2d_45", "tf__conv_46/conv2d_46"),
    ], source)
    assign_conv(model.conv7, "model/tf__conv_47/sequential_10/conv2d_47", source)
    assign_c3(model.c3_6, "model/tf__c3_6", "tf__conv_48/conv2d_48", "tf__conv_49/conv2d_49", "tf__conv_50/conv2d_50", [
        ("sequential_11/tf__bottleneck_10", "tf__conv_51/conv2d_51", "tf__conv_52/conv2d_52"),
    ], source)
    assign_conv(model.conv8, "model/tf__conv_53/sequential_12/conv2d_53", source)
    assign_c3(model.c3_7, "model/tf__c3_7", "tf__conv_54/conv2d_54", "tf__conv_55/conv2d_55", "tf__conv_56/conv2d_56", [
        ("sequential_13/tf__bottleneck_11", "tf__conv_57/conv2d_57", "tf__conv_58/conv2d_58"),
    ], source)
    assign_detect(model.detect.m[0], "model/tf__detect/tf__conv2d/conv2d_59", source)
    assign_detect(model.detect.m[1], "model/tf__detect/tf__conv2d_1/conv2d_60", source)
    assign_detect(model.detect.m[2], "model/tf__detect/tf__conv2d_2/conv2d_61", source)
    anchors = np.stack([
        source['model/tf__detect/mul_4'].reshape(3, 2),
        source['model/tf__detect/mul_11'].reshape(3, 2),
        source['model/tf__detect/mul_18'].reshape(3, 2),
    ], axis=0) * 320.0
    strides = np.array([
        source['model/tf__detect/mul_2'][0] * 320.0,
        source['model/tf__detect/mul_9'][0] * 320.0,
        source['model/tf__detect/mul_16'][0] * 320.0,
    ], dtype=np.float32)
    model.detect.anchors.copy_(torch.from_numpy(anchors.astype(np.float32)))
    model.detect.anchor_grid.copy_(model.detect.anchors.view(model.detect.nl, 1, model.detect.na, 1, 1, 2))
    model.detect.stride.copy_(torch.from_numpy(strides))


## 11. Instantiate the PyTorch model


In [ ]:
model = PuryfiYoloV5(num_classes=len(MODEL_LABEL_KEYS))
load_puryfi_weights(model, TF_WEIGHTS)
model.eval()
print("Model loaded. Total parameters:", sum(p.numel() for p in model.parameters()))


## 12. Define preprocessing and postprocessing utilities

We add helpers for image letterboxing, coordinate scaling, non-maximum suppression, and detection dataclasses.


In [ ]:
@dataclass
class Detection:
    label: str
    score: float
    box: Tuple[int, int, int, int]


def letterbox(image: Image.Image, size: int = 320, color: Tuple[int, int, int] = (114, 114, 114)) -> Tuple[Image.Image, float, Tuple[int, int]]:
    w, h = image.size
    scale = min(size / w, size / h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    resized = image.resize((new_w, new_h), Image.BILINEAR)
    canvas = Image.new("RGB", (size, size), color)
    pad_x = (size - new_w) // 2
    pad_y = (size - new_h) // 2
    canvas.paste(resized, (pad_x, pad_y))
    return canvas, scale, (pad_x, pad_y)


def prepare_tensor(image: Image.Image) -> torch.Tensor:
    array = np.asarray(image).astype(np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)
    return tensor


def box_iou(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    return inter / (area1[:, None] + area2 - inter + 1e-6)


def non_max_suppression(boxes: torch.Tensor, scores: torch.Tensor, iou_thresh: float) -> torch.Tensor:
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        i = order[0]
        keep.append(i.item())
        if order.numel() == 1:
            break
        iou = box_iou(boxes[i].unsqueeze(0), boxes[order[1:]])[0]
        order = order[1:][iou <= iou_thresh]
    return torch.tensor(keep, dtype=torch.long)


def scale_boxes(boxes: torch.Tensor, scale: float, pad: Tuple[int, int], original_size: Tuple[int, int]) -> torch.Tensor:
    boxes = boxes.clone()
    boxes[:, [0, 2]] -= pad[0]
    boxes[:, [1, 3]] -= pad[1]
    boxes /= scale
    boxes[:, 0].clamp_(0, original_size[0])
    boxes[:, 2].clamp_(0, original_size[0])
    boxes[:, 1].clamp_(0, original_size[1])
    boxes[:, 3].clamp_(0, original_size[1])
    return boxes


def run_detector(image: Image.Image, conf_thresh: float = 0.25, iou_thresh: float = 0.45, device: Optional[torch.device] = None) -> List[Detection]:
    device = device or torch.device("cpu")
    resized, scale, pad = letterbox(image)
    tensor = prepare_tensor(resized).to(device)
    model.to(device)
    with torch.no_grad():
        outputs = model(tensor)[0]
    boxes_xywh = outputs[:, :4]
    boxes_xyxy = torch.zeros_like(boxes_xywh)
    boxes_xyxy[:, 0] = boxes_xywh[:, 0] - boxes_xywh[:, 2] / 2
    boxes_xyxy[:, 1] = boxes_xywh[:, 1] - boxes_xywh[:, 3] / 2
    boxes_xyxy[:, 2] = boxes_xywh[:, 0] + boxes_xywh[:, 2] / 2
    boxes_xyxy[:, 3] = boxes_xywh[:, 1] + boxes_xywh[:, 3] / 2
    objectness = outputs[:, 4]
    class_scores = outputs[:, 5:]
    scores, labels = (objectness.unsqueeze(1) * class_scores).max(dim=1)
    mask = scores > conf_thresh
    boxes_xyxy = boxes_xyxy[mask]
    scores = scores[mask]
    labels = labels[mask]
    if boxes_xyxy.numel() == 0:
        return []
    boxes_xyxy = scale_boxes(boxes_xyxy, scale, pad, image.size)
    detections: List[Detection] = []
    for class_idx in labels.unique():
        class_mask = labels == class_idx
        class_boxes = boxes_xyxy[class_mask]
        class_scores = scores[class_mask]
        keep = non_max_suppression(class_boxes, class_scores, iou_thresh)
        for idx in keep:
            box = class_boxes[idx].round().int().tolist()
            detections.append(
                Detection(
                    label=MODEL_LABEL_KEYS[int(class_idx)],
                    score=float(class_scores[idx]),
                    box=(box[0], box[1], box[2], box[3]),
                )
            )
    return detections


## 13. Apply censor effects

The helper below can blur, pixelate, or paint over detected regions using Pillow.


In [ ]:
def apply_pixelate(region: Image.Image, pixels: int = 12) -> Image.Image:
    w, h = region.size
    small = region.resize((max(1, w // pixels), max(1, h // pixels)), Image.NEAREST)
    return small.resize((w, h), Image.NEAREST)


def apply_censor(image: Image.Image, detections: Iterable[Detection], mode: str = "pixelate", color: Tuple[int, int, int] = (0, 0, 0)) -> Image.Image:
    out = image.copy()
    for detection in detections:
        box = detection.box
        region = out.crop(box)
        if mode == "pixelate":
            censored = apply_pixelate(region)
        elif mode == "blur":
            censored = region.filter(ImageFilter.GaussianBlur(radius=12))
        elif mode == "solid":
            censored = Image.new("RGB", region.size, color)
        else:
            raise ValueError(f"Unsupported censor mode: {mode}")
        out.paste(censored, box)
    return out


def draw_detections(image: Image.Image, detections: Iterable[Detection]) -> Image.Image:
    out = image.copy()
    draw = ImageDraw.Draw(out)
    for det in detections:
        draw.rectangle(det.box, outline="red", width=2)
        draw.text((det.box[0] + 2, det.box[1] + 2), f"{det.label} {det.score:.2f}", fill="red")
    return out


## 14. Batch processing helper

This utility scans a directory of images, runs the detector, applies the chosen censor mode, and writes censored copies to an output directory.


In [ ]:
def process_directory(
    input_dir: Path,
    output_dir: Path,
    mode: str = "pixelate",
    conf_thresh: float = 0.25,
    iou_thresh: float = 0.45,
    label_filter: Optional[Sequence[str]] = None,
    device: Optional[torch.device] = None,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    valid_suffixes = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    files = [p for p in sorted(input_dir.iterdir()) if p.suffix.lower() in valid_suffixes]
    for path in tqdm(files, desc="Censoring images"):
        image = Image.open(path).convert("RGB")
        detections = run_detector(image, conf_thresh=conf_thresh, iou_thresh=iou_thresh, device=device)
        if label_filter:
            detections = [det for det in detections if det.label in label_filter]
        censored = apply_censor(image, detections, mode=mode)
        censored.save(output_dir / path.name)


## 15. Configure batch parameters

Update the paths below to point to a directory of input images. The notebook writes censored images to `output_dir`.


In [ ]:
input_dir = BASE_DIR / "sample_images"
output_dir = BASE_DIR / "censored_output"
selected_labels = MODEL_LABEL_KEYS  # Replace with a subset to censor specific classes
censor_mode = "pixelate"  # Choose from: "pixelate", "blur", "solid"
confidence_threshold = 0.35
iou_threshold = 0.45
print("Input directory:", input_dir)
print("Output directory:", output_dir)


## 16. Preview a single image

Run this cell to inspect detector hits and the censored result before launching the full batch job.


In [ ]:
preview_path = next(iter(sorted(input_dir.glob("*"))), None)
if preview_path is None:
    print("Populate", input_dir, "with sample images before running the preview.")
else:
    original = Image.open(preview_path).convert("RGB")
    preview_detections = run_detector(original, conf_thresh=confidence_threshold, iou_thresh=iou_threshold)
    if selected_labels:
        preview_detections = [det for det in preview_detections if det.label in selected_labels]
    print(f"Detections on {preview_path.name}:")
    for det in preview_detections:
        print(f" - {det.label}: {det.score:.3f} @ {det.box}")
    display(draw_detections(original, preview_detections))
    display(apply_censor(original, preview_detections, mode=censor_mode))


## 17. Run the batch censoring job

Uncomment the call below to process the entire directory once you are satisfied with the preview.


In [ ]:
# process_directory(
#     input_dir=input_dir,
#     output_dir=output_dir,
#     mode=censor_mode,
#     conf_thresh=confidence_threshold,
#     iou_thresh=iou_threshold,
#     label_filter=selected_labels,
# )
